# ATiG 2026: LT-FH exercise using ltpred

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bvilhjal/ATIG_2026/blob/main/teaching_days/2026-09-24/LTFH/LTFH_exercise.ipynb)

## What you will learn

You score a simulated register with [ltpred](https://github.com/bvilhjal/ltpred)'s
family-history method and compare the scores with the true genetic liability, which a
simulation lets you see. Five parts, 15 questions:

- **In class (about an hour):** Part 0 (10 min) sets up; **A** (20 min): a wrong
  heritability h² rescales the score but barely changes its ranking; **B** (30 min): the
  score is not a risk, so how well does it predict who is diagnosed later?
- **Homework:** **C** the score against a simple yes/no family-history indicator;
  **D** estimating h² from parents and children; **E** the two scales on which h² is
  reported.

Work in pairs, ideally one person with a biology and one with a bioinformatics
background: questions marked **Discuss** are for talking through together. Before each
part, write down what you expect.

**Background.** LT-FH ([Hujoel et al. 2020](https://doi.org/10.1038/s41588-020-0613-6))
turns relatives' diagnoses into an estimate of a person's genetic liability;
LT-FH++ ([Pedersen et al. 2022](https://doi.org/10.1016/j.ajhg.2022.01.009)) adds age of
onset. ltpred computes it with the Pearson–Aitken method of PA-FGRS
([Krebs et al. 2024](https://doi.org/10.1016/j.ajhg.2024.09.009)). The model is on the
[algorithm page](https://bvilhjal.github.io/ltpred/algorithm/).

## The idea in five sentences

1. Everyone has a **liability** to the disease: an unobserved quantity that adds up a
   genetic part and an environmental part and follows a bell curve in the population.
2. People whose liability passes a **threshold** get the disease. The threshold falls with
   age, so the higher your liability, the earlier you are diagnosed.
3. **Heritability h²** is the share of liability variance that is genetic. Relatives share
   genes, so their diagnoses carry information about your genetic liability.
4. **LT-FH** turns the relatives' diagnoses and ages into a best estimate of your genetic
   liability (the *score*), with a measure of its uncertainty. The score can replace a
   yes/no phenotype in a GWAS, or feed a risk prediction.
5. Real data never reveal the true genetic liability. A simulation does, so here you can
   check how good the score is.

| Term | Meaning here |
|---|---|
| liability | the unobserved disease predisposition; genetic part `g` plus the rest |
| h² (liability scale) | share of liability variance due to additive genes; 0.5 here |
| K, prevalence | fraction who get the disease in a lifetime; 0.10 here |
| cumulative incidence (CIP) | fraction diagnosed by a given age; rises from 0 to K |
| score, posterior mean | best estimate of `g` given the family's records |
| posterior variance | how uncertain that estimate is |
| AUC | chance that a random case scores higher than a random non-case (0.5 = coin flip) |
| calibration slope | slope of the truth regressed on the score; 1 means the score's scale is right |
| tetrachoric correlation | correlation of two liabilities, inferred from two yes/no variables |
| observed scale | h² measured on the 0/1 diagnosis itself instead of on the liability |

## How to use this notebook

1. Click **Open in Colab** above, then **File → Save a copy in Drive** and work in your copy.
2. Run the cells in order with **Shift+Enter**. If Colab restarts, start again from the top.
3. Where a cell has `...`, replace it with one expression; the comment says what.
   **Checkpoints** tell you what you should see.

The names you will use throughout:

| Name | What it is |
|---|---|
| `reg` | the simulated register (one row per person) |
| `g` | each person's true genetic liability |
| `est`, `var` | everyone's score and its posterior variance, using all records |
| `free40` | True for people still undiagnosed at 40 |
| `est40`, `var40` | the score as known at 40, for those people |
| `score(h2)`, `score_at_40(h2)` | rescore the register under any assumed h² |

### The Python you need

New to Python? These four patterns cover every blank. Each task also has a **Hint**.

| Code | What it does |
|---|---|
| `x[mask]` | keeps the entries of array `x` where the True/False array `mask` is True |
| `a & b` | True where both `a` and `b` are True |
| `x.mean()`, `x.sum()` | average and total; on True/False arrays, the share and count of True |
| `x ** 2` | squares every entry |

## Setup

The first cell installs ltpred; the second loads it and defines two helpers, `corr` and
`auc`. (On your own computer instead of Colab? See the
[setup page](https://bvilhjal.github.io/ATIG_2026/setup.html).)

In [ ]:
%pip install -q git+https://github.com/bvilhjal/ltpred@v0.7.1     # installs ltpred (about 30 s)

In [1]:
import numpy as np
from scipy.stats import norm, rankdata

import ltpred
from ltpred import simulate_pedigree, simulate_register_liabilities, estimate_liabilities
print("ltpred", ltpred.__version__)


def corr(x, y):
    """Pearson correlation of two arrays."""
    return np.corrcoef(x, y)[0, 1]


def auc(score, case):
    """AUC: the chance that a random case scores above a random non-case.

    Computed from ranks (the Mann-Whitney statistic); `case` is a boolean array."""
    r = rankdata(score)
    n1, n0 = case.sum(), (~case).sum()
    return (r[case].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

ltpred 0.7.1


## Part 0. A register where the truth is known

The cell below simulates three generations of families (5,075 people) and gives each
person a liability: a genetic part `g` with variance h² = 0.5 plus a non-genetic part.
A person is diagnosed at the age their liability crosses an age-dependent threshold, so
high liability means early onset. Everyone is followed to age 70. This is the model of
the [ltpred tutorial](https://bvilhjal.github.io/ltpred/tutorial/), five times larger.

`reg` holds what a real register would, plus the truth:

| Field | Meaning |
|---|---|
| `reg.ids`, `reg.father`, `reg.mother` | the pedigree |
| `reg.status` | diagnosed by age 70 (True/False) |
| `reg.age` | age at diagnosis, or 70 for the undiagnosed |
| `reg.onset`, `reg.birth_time` | age at onset (`inf` if never) and calendar time of birth |
| `reg.genetic` | the true genetic liability, stored as `g` |

In [2]:
H2 = 0.5                                    # true liability-scale heritability
K = 0.10                                    # lifetime prevalence
AGES = np.arange(0, 121.0)                  # ages 0, 1, ..., 120
CIP = K / (1 + np.exp((60 - AGES) / 8))     # cumulative incidence by age: half of K by age 60

ids, father, mother = simulate_pedigree(np.random.default_rng(1), n_founder_pairs=500, gens=2)
reg = simulate_register_liabilities(np.random.default_rng(1), ids, father, mother,
                                    h2=H2, cip_ages=AGES, cip_values=CIP, eval_age=70)
g = reg.genetic                             # the truth, known only because we simulated it

print(f"{len(reg.ids)} people, {reg.status.sum()} diagnosed by age 70 ({reg.status.mean():.1%})")
print(f"var(g) = {g.var():.3f}   (target {H2})")

5075 people, 366 diagnosed by age 70 (7.2%)
var(g) = 0.493   (target 0.5)


**Checkpoint.** 5,075 people, 366 diagnosed (7.2%), and var(g) close to 0.5.

### The scorer in one call

`estimate_liabilities` takes the pedigree, everyone's diagnosis and age, the incidence
curve and h², and returns for each proband the **posterior mean** of their genetic
liability given the relatives' records (`.est`), with its **posterior variance**
(`.var`). The comments explain each argument.

In [3]:
scores = estimate_liabilities(
    reg.ids, reg.father, reg.mother,          # the pedigree: who is whose parent
    probands=reg.ids,                         # whom to score: everyone
    status=reg.status, age=reg.age,           # diagnosed by 70? age at diagnosis or at 70
    cip_ages=AGES, cip_values=CIP, k_pop=K,   # incidence curve: turns an age into a liability threshold
    h2=H2,                                    # heritability: you supply it, the scorer never estimates it
    use="gwas")                               # also use each person's own diagnosis

est, var = scores.est, scores.var
print(f"corr(score, g) = {corr(est, g):.3f}")
print(f"var(score) = {est.var():.3f}  +  mean posterior variance = {var.mean():.3f}"
      f"  =  {est.var() + var.mean():.3f}")

corr(score, g) = 0.522
var(score) = 0.129  +  mean posterior variance = 0.359  =  0.488


**Checkpoint.** Correlation 0.522. The variance of the scores plus the average posterior
variance comes back to about var(g): the data explain part of each person's genetic
liability and the posterior variance holds the rest. Part A asks what happens to this
sum when h² is wrong.

### Two helpers used in every question

`score(h2)` repeats the call above under any assumed h². `score_at_40(h2)` scores the
people still undiagnosed at 40 **as of their 40th birthday**: `use="prediction"` hides
their own status and every record made after that date. This is the honest setting for
predicting who is diagnosed later.

In [4]:
free40 = reg.onset > 40                     # still undiagnosed on their 40th birthday


def score(h2):
    """Everyone's score from all records up to 70 (use="gwas"): (mean, variance)."""
    s = estimate_liabilities(reg.ids, reg.father, reg.mother, probands=reg.ids,
                             status=reg.status, age=reg.age,
                             cip_ages=AGES, cip_values=CIP, k_pop=K, h2=h2, use="gwas")
    return s.est, s.var


def score_at_40(h2):
    """Scores as known on each person's 40th birthday (use="prediction"), for the
    people still undiagnosed then: their own status and every later record are hidden."""
    s = estimate_liabilities(reg.ids, reg.father, reg.mother,
                             probands=[p for p, keep in zip(reg.ids, free40) if keep],
                             status=reg.status, age=reg.age,
                             cip_ages=AGES, cip_values=CIP, k_pop=K, h2=h2, use="prediction",
                             birth_time=reg.birth_time,                  # everyone's birth date
                             index_time=(reg.birth_time + 40)[free40])   # each proband's 40th birthday
    return s.est, s.var


est40, var40 = score_at_40(H2)
print(f"{free40.sum()} people undiagnosed at 40; corr(score at 40, g) = {corr(est40, g[free40]):.3f}")

5037 people undiagnosed at 40; corr(score at 40, g) = 0.278


**Checkpoint.** 5,037 people are undiagnosed at 40, and their score at 40 correlates
0.278 with `g`: weaker than 0.522, because it knows less.

## Part A. What does a wrong h² do?

In a real analysis h² comes from outside (a twin, pedigree or SNP study), on the
liability scale, and ltpred makes you supply it. Here you can pass a wrong value on
purpose and compare with the truth.

### Q1: If you tell the scorer h² = 0.8 when the truth is 0.5, do the scores spread more or less? Does their ranking change?

Write your expectation with one sentence of reasoning. Hint: h² is the share of
liability variance that is genetic, and therefore shared with relatives.

**Discuss** with your partner.

### Q2: Score the register under h² = 0.2, 0.5 and 0.8.

For each value the loop prints the correlation with `g`, the variance of the scores, the
mean posterior variance, their sum, and the **calibration slope**: the slope of `g`
regressed on the score. Fill in the two `...`.

In [ ]:
print("assumed h2   corr  var(score)  mean var    sum  slope")
for h2 in (0.2, 0.5, 0.8):
    e, v = score(h2)
    c = ...          # correlation of e with the truth g
    b = ...          # slope of g regressed on e: np.polyfit(e, g, 1)[0]
    print(f"{h2:10.1f} {c:6.3f} {e.var():10.3f} {v.mean():9.3f} {e.var() + v.mean():6.3f} {b:6.2f}")

<details><summary><b>Hint</b></summary>

The helpers from Setup do the work: `corr(x, y)` is a correlation, and `np.polyfit(x, y, 1)[0]` is the slope of `y` regressed on `x`. Here `x` is the score `e` and `y` is the truth `g`.

</details>

**Checkpoint.** The 0.5 row repeats Part 0. The `sum` column comes out close to the h²
you passed in, whichever it was, and one column barely moves at all.

### Q3: Which columns moved and which did not? What does a slope of 1 mean, and why does only the right h² give it?

A score is *calibrated* when the truth regressed on it has slope 1: among people scored
0.3, the true liability averages 0.3.

**Discuss** with your partner.

## Part B. Who is diagnosed between 40 and 70?

![Liability thresholds at 40 and 70. Higher liability is earlier onset. People still undiagnosed at 40 are left of T40; those diagnosed by 70 have crossed T70.](https://bvilhjal.github.io/ATIG_2026/teaching_days/2026-09-24/LTFH/figures/thresholds.png)

A correlation with `g` is not the accuracy of predicting a later diagnosis. Here you
measure that accuracy directly, using the score at 40 (`est40`) and what happened next.

### Q4: How many of the people undiagnosed at 40 are diagnosed by 70?

`reg.status` means "diagnosed by 70" and `free40` means "undiagnosed at 40". The result
`y` must line up with `est40`, which has one entry per person in `free40`.

In [ ]:
y = ...          # one boolean per person in est40: diagnosed between 40 and 70
print(f"{y.sum()} incident cases among {len(y)} people ({y.mean():.1%})")

<details><summary><b>Hint</b></summary>

Keep the entries of `reg.status` for the people in `free40` (the `x[mask]` pattern). Everyone in `free40` was undiagnosed at 40, so diagnosed by 70 means diagnosed between 40 and 70.

</details>

**Checkpoint.** A few hundred cases among 5,037 people, an incidence a little below the
register's 7.2%.

### Q5: Compute the AUC of three predictors of that outcome, on the same people.

The AUC is the chance that a random case scores above a random non-case (0.5 is a coin
flip, 1 is perfect). Compare: the score at 40, `est40`; the true genetic liability `g`,
the best any genetic predictor could do; and the `use="gwas"` score `est`, which saw each
person's own diagnosis. **Before running**, order the three.

In [ ]:
print(f"AUC score at 40             {auc(est40, y):.3f}")
print(f"AUC true g                  {auc(..., y):.3f}")      # g for the same people
print(f"AUC use='gwas' score        {auc(..., y):.3f}")      # est for the same people

<details><summary><b>Hint</b></summary>

Select the same people from `g` and from `est` with `[free40]`, so that all three predictors line up with `y`.

</details>

**Checkpoint.** The `use="gwas"` score sits near 1; if not, you did not restrict it to the
people in `free40`. The other two are clearly above 0.5 and clearly apart.

### Q6: Why is even the truth below AUC 1? Why is the score at 40 so far below the truth? What does the third AUC say about using a `use="gwas"` score for prediction?

The cell below prints three variances that help.

**Discuss** with your partner.

In [5]:
print(f"var(g)                        {g.var():.3f}")
print(f"var(score), use='gwas'        {est.var():.3f}")
print(f"var(score at 40)              {est40.var():.3f}")

var(g)                        0.493
var(score), use='gwas'        0.129
var(score at 40)              0.040


### Q7: Turn the score at 40 into a risk and check it against what happened.

`est40` is not a probability. Given the relatives, a person's full liability is normal
with mean `est40` and standard deviation `sd = sqrt(var40 + 1 − h²)`. Being undiagnosed
at 40 means it is below the threshold T40; diagnosis by 70 means it is above T70. So

**risk = [P(below T40) − P(below T70)] / P(below T40)**,

where P(below T) = Φ((T − est40) / sd). The first lines compute both probabilities,
`below_T40` and `below_T70`; write the risk. The loop then compares risk with the observed rate within
fifths of the score.

In [ ]:
T40, T70 = norm.isf(np.interp([40, 70], AGES, CIP))   # liability thresholds at ages 40 and 70
sd = np.sqrt(var40 + 1 - H2)                          # spread of full liability given the relatives
below_T40 = norm.cdf((T40 - est40) / sd)              # P(liability < T40): undiagnosed at 40
below_T70 = norm.cdf((T70 - est40) / sd)              # P(liability < T70): undiagnosed at 70
risk = ...                                            # the formula above

print(f"mean risk {risk.mean():.3f}   observed incidence {y.mean():.3f}")
fifths = np.array_split(np.argsort(est40, kind="stable"), 5)    # lowest to highest score
for name, idx in zip(("lowest", "second", "middle", "fourth", "highest"), fifths):
    print(f"{name:>8} fifth: {y[idx].sum():3d} cases, observed {y[idx].mean():.3f}, "
          f"risk {risk[idx].mean():.3f}")

<details><summary><b>Hint</b></summary>

Put the two variables `below_T40` and `below_T70` into the formula.

</details>

**Checkpoint.** The mean risk is within about half a percentage point of the observed
incidence, and the lowest and highest fifths clearly differ.

## Part C. The score against a 0/1 family-history indicator (homework)

A common shortcut is the yes/no indicator "any parent or full sibling diagnosed". The
cell below finds each person's parents and full siblings and builds that indicator twice:
with follow-up to 70 (`fh`) and as known on each person's 40th birthday (`fh40`). Run it
as is.

In [6]:
row = {p: i for i, p in enumerate(reg.ids)}          # id -> row number
children = {}                                        # (father, mother) -> rows of their children
for i, (f, m) in enumerate(zip(reg.father, reg.mother)):
    if f in row and m in row:
        children.setdefault((f, m), []).append(i)


def first_degree(i):
    """Rows of person i's parents and full siblings."""
    f, m = reg.father[i], reg.mother[i]
    parents = [row[p] for p in (f, m) if p in row]
    siblings = [j for j in children.get((f, m), []) if j != i]
    return np.array(parents + siblings, dtype=int)


fdr = [first_degree(i) for i in range(len(reg.ids))]
diag_time = reg.birth_time + reg.onset               # calendar time of each diagnosis
birth40 = reg.birth_time + 40                        # calendar time of each 40th birthday

# yes/no family history: any affected parent or sibling, by 70 and by one's own 40th birthday
fh = np.array([reg.status[r].any() for r in fdr])
fh40 = np.array([(reg.status[r] & (diag_time[r] <= birth40[i])).any() for i, r in enumerate(fdr)])
print(f"family-history positive: {fh.sum()} by age 70, {fh40.sum()} at their 40th birthday")

family-history positive: 954 by age 70, 609 at their 40th birthday


### Q8: How much does the score improve on the indicator?

First, how much of `g` does each explain (R², the squared correlation): own status, the
indicator `fh`, and the `use="gwas"` score? Then, as a prediction: the AUC of `fh40`
against the score at 40 for Part B's outcome. Why is own status left out of the second
comparison?

In [ ]:
for name, x in (("own status", reg.status), ("FH indicator 0/1", fh), ("use='gwas' score", est)):
    print(f"R2  {name:18s} {...:.3f}")            # squared correlation with g
for name, x in (("FH indicator at 40", fh40[free40]), ("score at 40", est40)):
    print(f"AUC {name:18s} {auc(x, y):.3f}")

<details><summary><b>Hint</b></summary>

R² is the squared correlation: `corr(x, g) ** 2`.

</details>

**Checkpoint.** With follow-up to 70 the score's R² beats the indicator's clearly.
Prospectively the gap is much smaller.

## Part D. Where does h² come from? (homework)

Part A showed that h² matters. A quick cross-check: h² is about twice the **tetrachoric
correlation** between parents' and children's diagnoses, the correlation of the
underlying liabilities inferred from a 2×2 table. The cell below lists every
parent–child pair as row numbers `par` and `kid` and defines `table`; run it as is.

In [7]:
from ltpred import tetrachoric

# every parent-child pair in the register, as row numbers
pairs = [(row[p], i) for i, (f, m) in enumerate(zip(reg.father, reg.mother))
         for p in (f, m) if p in row]
par, kid = np.array(pairs).T


def table(a, b):
    """2x2 table of two True/False arrays: both, first only, second only, neither."""
    return np.array([(a & b).sum(), (a & ~b).sum(), (~a & b).sum(), (~a & ~b).sum()])

### Q9: Estimate h² from the parent–offspring pairs.

In [ ]:
p_status, k_status = reg.status[par], reg.status[kid]
t = ...                      # tetrachoric(first status array, second status array)
print(f"{len(par)} pairs, table {table(p_status, k_status)}")
print(f"h2 = 2 rho = {2 * t.rho:.2f} +/- {2 * t.se:.2f}   (truth {H2})")

<details><summary><b>Hint</b></summary>

`tetrachoric` takes the two True/False arrays: the parents' statuses and the children's.

</details>

**Checkpoint.** An estimate within one standard error of 0.5, and a small first cell
(pairs where both are diagnosed).

### Q10: Why does twice the tetrachoric work here, and what would break it in a real register?

Think about the threshold, shared environment, and how the register was sampled.

**Discuss** with your partner.

## Part E. Which scale is h² on? (homework)

![A liability split into a 0/1 outcome. The right tail is the cases, 10% when K = 0.10. Everyone on one side receives the same y.](https://bvilhjal.github.io/ATIG_2026/teaching_days/2026-09-24/LTFH/figures/observed_scale.png)

GWAS methods such as GREML and LD-score regression report h² on the observed 0/1 scale,
and that number depends on the sample's case proportion P. With prevalence K and
z = φ(Φ⁻¹(1 − K)) ([Lee et al. 2011](https://doi.org/10.1016/j.ajhg.2011.02.002)):

**h²_liab = h²_obs · K²(1 − K)² / (z² · P(1 − P))**,

with P = K in a population sample. ltpred's `liability_to_observed_h2` and
`observed_to_liability_h2` apply this formula and its inverse.

### Q11: What observed-scale h² would each design report?

The truth is h² = 0.5 with K = 0.10. Compute the observed-scale value for a population
sample (`P=None`), a 1:1 case/control study (P = 0.5) and a 1:4 study (P = 0.2), and
convert each back.

In [ ]:
from ltpred import liability_to_observed_h2, observed_to_liability_h2

for name, P in (("population sample", None), ("1:1 case/control", 0.5), ("1:4 case/control", 0.2)):
    obs = ...        # liability_to_observed_h2(H2, K, P)
    back = ...       # observed_to_liability_h2(obs, K, P)
    print(f"{name:18s} h2_obs = {float(obs):.3f}   converted back = {float(back):.3f}")

<details><summary><b>Hint</b></summary>

The comments give each call; `P` is set by the loop.

</details>

**Checkpoint.** All three convert back to 0.500, and the observed values differ by more
than a factor of two.

### Q12: Analysis note.

In at most 100 words: what does the family-history score estimate, what did it show in
this exercise, and what can it not establish? Cite one number from each part you did.

**Discuss** with your partner.

---
Part of [ATIG 2026](https://github.com/bvilhjal/ATIG_2026), teaching day
[24 September](https://github.com/bvilhjal/ATIG_2026/tree/main/teaching_days/2026-09-24).